# RWGNN training + inference from saved XY BLT splits

This notebook reuses the scripted pipeline (`RWGNN/train.py` and `inference.py`) but runs everything inline.
It loads the generated `train.npz`/`val.npz`/`test.npz` splits (see `XYModel/generate_blt_dataset.py`),
trains the Random Walk GNN, visualises metrics, and previews Tc interval predictions on held-out lattices.


In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt

from RWGNN.data import load_xy_splits
from RWGNN.model import RandomWalkGNN
from RWGNN.train import _make_loaders, train_epoch, evaluate
from inference import _preview_predictions, _load_model

# Ensure deterministic-ish behaviour for the quick demo
torch.manual_seed(7)
np.random.seed(7)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
# Paths and hyperparameters aligning with train.py / inference.py
dataset_dir = Path('XYModel/blt_dataset')  # expects train.npz, val.npz, test.npz, metadata.json
checkpoint_path = Path('artifacts/rwg_nn_demo.pt')

walk_length = 4
num_walks = 8
batch_size = 8
hidden_channels = 128
epochs = 6  # keep short for the demo
learning_rate = 2e-3


In [ ]:
# Load the generated splits and build loaders using the scripted helper
splits = load_xy_splits(dataset_dir, walk_length=walk_length, num_walks=num_walks)
feature_size = splits.train[0].num_node_features
train_loader, val_loader, test_loader = _make_loaders(splits, batch_size)
feature_size


In [ ]:
# Model and optimizer mirror train.py
model = RandomWalkGNN(in_channels=feature_size, hidden_channels=hidden_channels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
model


In [ ]:
# Train with the same helpers used by train.py while recording metrics for plotting
history = {
    'train_loss': [],
    'val_acc': [],
    'val_temp_mae': [],
    'val_tc_mae': [],
}
for epoch in range(1, epochs + 1):
    train_loss = train_epoch(model, train_loader, optimizer, device)
    val_metrics = evaluate(model, val_loader, device)
    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_metrics.phase_acc)
    history['val_temp_mae'].append(val_metrics.temp_mae)
    history['val_tc_mae'].append(val_metrics.tc_mae)
    print(f'Epoch {epoch:02d} | train_loss={train_loss:.4f} | '
          f'val_acc={val_metrics.phase_acc:.3f} | '
          f'val_temp_mae={val_metrics.temp_mae:.3f} | val_tc_mae={val_metrics.tc_mae:.3f}')


In [ ]:
# Visualise validation metrics
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(history['train_loss'], label='train')
axes[0].set_title('Training loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history['val_acc'], label='val acc')
axes[1].set_title('Validation phase accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

axes[2].plot(history['val_tc_mae'], label='val Tc MAE')
axes[2].plot(history['val_temp_mae'], label='val temp MAE')
axes[2].set_title('Validation regression errors')
axes[2].set_xlabel('Epoch')
axes[2].legend()
plt.tight_layout()
plt.show()


In [ ]:
# Evaluate on the held-out test split using the same evaluator as inference.py
test_metrics = evaluate(model, test_loader, device)
print(json.dumps({
    'test_acc': test_metrics.phase_acc,
    'test_temp_mae': test_metrics.temp_mae,
    'test_tc_mae': test_metrics.tc_mae,
}, indent=2))


In [ ]:
# Plot predicted vs true phases on the full test split
model.eval()
true_phases, pred_phases = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        outputs = model(batch)
        preds = outputs['phase_logits'].argmax(dim=-1).cpu().numpy()
        pred_phases.append(preds)
        true_phases.append(batch.y.view(-1).cpu().numpy())

true_phases = np.concatenate(true_phases) if true_phases else np.array([])
pred_phases = np.concatenate(pred_phases) if pred_phases else np.array([])

classes = ['ordered', 'disordered']
cm = np.zeros((len(classes), len(classes)), dtype=int)
for t, p in zip(true_phases, pred_phases):
    cm[int(t), int(p)] += 1

fig, ax = plt.subplots(figsize=(4.5, 4.0))
im = ax.imshow(cm, cmap='Blues')
ax.set_xlabel('Predicted phase')
ax.set_ylabel('True phase')
ax.set_xticks(range(len(classes)))
ax.set_yticks(range(len(classes)))
ax.set_xticklabels(classes)
ax.set_yticklabels(classes)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='black')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.title('True vs predicted BLT phase on test set')
plt.tight_layout()
cm


In [ ]:
# Preview predictions and Tc intervals exactly as inference.py does
_preview_predictions(model, test_loader, device, max_batches=1)


In [ ]:
# Save the trained weights and demonstrate loading with the inference helper
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({'model_state': model.state_dict(), 'in_channels': feature_size}, checkpoint_path)
print(f'Saved checkpoint to {checkpoint_path}')

reloaded = _load_model(checkpoint_path, feature_size, hidden_channels, device)
_preview_predictions(reloaded, test_loader, device, max_batches=1)
